# DPX Intelligence — Paying for Signals with x402

**DPX Intelligence** is a standalone signal API. Agents pay micropayments via x402 to access structured intelligence on macro conditions, climate systems, FX corridors, cascade risk, supply chain stress, sovereign debt, and more.

This notebook shows how to:
- Call any DPX Intelligence endpoint
- Handle the `402 Payment Required` response
- Sign an EIP-3009 transfer and pay $0.15–$0.75 USDC on Base
- Receive and reason about the structured signal

**DPX Intelligence is a separate product from DPX Settlement.** Signals can be used for any purpose — investment research, risk management, autonomous agent decision-making, or data pipelines. The final section of this notebook shows one example of using intelligence signals alongside DPX Settlement, but settlement is not required.

## x402 payment flow

```
1. GET /v1/intelligence/macro-stress
   → 402: { accepts: [{ amount: '150000', asset: USDC, payTo: '0x...' }] }

2. Agent signs EIP-3009 TransferWithAuthorization
   → X-Payment: <signed token>

3. GET /v1/intelligence/macro-stress  (retry with X-Payment header)
   → 200: { regime: 'ELEVATED_RISK', score: 72, ... }
```

No intermediary holds funds. The transfer authorizes USDC to move from the agent's wallet to DPX on-chain — the response is delivered once the transfer is verified.

**Mock mode:** Set `AGENT_PRIVATE_KEY` in `.env` to pay real x402 fees on Base. Leave it blank to use simulated signals with no USDC spent.

In [ ]:
%pip install anthropic httpx python-dotenv eth-account --quiet

In [ ]:
import os, json, httpx, anthropic
from dotenv import load_dotenv
load_dotenv()

client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
INTEL  = 'https://intelligence.untitledfinancial.com'

# Set AGENT_PRIVATE_KEY in .env to pay real x402 fees (funded Base wallet)
# Leave blank for mock mode — simulated signals, no USDC spent
AGENT_PRIVATE_KEY = os.environ.get('AGENT_PRIVATE_KEY', '')
MOCK = not AGENT_PRIVATE_KEY

print(f'Intelligence mode: {"mock (no payment)" if MOCK else "live x402 (USDC on Base)"}')

## Step 1 — Browse the catalogue (free)

In [ ]:
# The endpoint catalogue requires no payment
catalogue = httpx.get(f'{INTEL}/').json()
print(f"DPX Intelligence — {len(catalogue.get('endpoints', []))} endpoints\n")
for ep in catalogue.get('endpoints', []):
    print(f"  {ep.get('price',''):8s}  {ep.get('path','').replace('/v1/intelligence/',''):30s}  {ep.get('label','')}")

## Step 2 — Probe an endpoint, inspect the 402

In [ ]:
# Call an endpoint without payment — expect a 402
r = httpx.get(f'{INTEL}/v1/intelligence/macro-stress')
print(f'Status: {r.status_code}')

if r.status_code == 402:
    req  = r.json()
    acpt = req['accepts'][0]
    print(f"\nPayment required:")
    print(f"  Amount : {int(acpt['amount'])/1e6:.4f} USDC")
    print(f"  Asset  : {acpt['asset'][:10]}... (USDC on Base)")
    print(f"  Pay to : {acpt['payTo'][:10]}...")
    print(f"  Network: {acpt['network']}")

## Step 3 — Buy a signal

In [ ]:
# Mock signals for development (used when AGENT_PRIVATE_KEY is not set)
MOCK_SIGNALS = {
    'macro-stress': {
        'regime': 'ELEVATED_RISK', 'score': 72, 'confidence': 0.87,
        'drivers': ['HY spread widening', 'TED spread 38bps', 'VIX 24'],
        'recommendation': 'Elevated credit stress — monitor HY spreads before committing',
        'outlook': 'Stress likely to persist 2–4 weeks absent Fed intervention',
        'horizon': '30d'
    },
    'sovereign-debt': {
        'riskTier': 'MODERATE', 'score': 65, 'confidence': 0.81,
        'flags': ['US deficit trajectory elevated', 'EM debt rollover stress Q3'],
        'recommendation': 'USD corridor stable; elevated caution for EM exposure >30d'
    },
    'fx-settlement': {
        'corridor': 'USD/USD', 'stability': 'OPTIMAL', 'confidence': 0.95,
        'executionRisk': 'LOW', 'volatility24h': '0.02%',
        'recommendation': 'Domestic USD corridor — conditions optimal'
    },
    'shipping-stress': {
        'globalStress': 'MODERATE', 'score': 58, 'confidence': 0.79,
        'hotspots': ['Red Sea rerouting adding 8–12d', 'LA port congestion 3d'],
        'invoiceDelayRisk': 'LOW for domestic USD payments'
    },
    'cascade': {
        'cascadeRisk': 'LOW', 'failureProbability': 0.04, 'confidence': 0.91,
        'cascadeDepth': 2, 'affectedCounterparties': 3,
        'recommendation': 'Cascade risk within acceptable bounds'
    },
    'climate': {
        'temperatureAnomaly': '+1.4°C', 'droughtIndex': 'MODERATE', 'confidence': 0.88,
        'wildfireRisk': 'ELEVATED', 'co2Trajectory': 'ABOVE TARGET',
        'recommendation': 'Climate conditions elevated — relevant for agricultural/energy exposure'
    },
}


def buy_intelligence(endpoint: str) -> dict:
    """Buy a DPX Intelligence signal via x402."""
    if MOCK:
        name   = endpoint.rstrip('/').split('/')[-1]
        signal = MOCK_SIGNALS.get(name, {'status': 'mock', 'note': f'Simulated signal for {name}'})
        print(f'  [mock] {endpoint}')
        return signal

    # ── Live x402 ────────────────────────────────────────────────────────────
    import time
    from eth_account import Account
    from eth_account.messages import encode_structured_data

    url = f'{INTEL}{endpoint}'
    r   = httpx.get(url, timeout=10)
    if r.status_code == 200:
        return r.json()
    if r.status_code != 402:
        return {'error': f'Unexpected status {r.status_code}'}

    acpt         = r.json()['accepts'][0]
    amount       = int(acpt['amount'])
    pay_to       = acpt['payTo']
    asset        = acpt['asset']
    account      = Account.from_key(AGENT_PRIVATE_KEY)
    valid_after  = int(time.time()) - 10
    valid_before = int(time.time()) + 300
    nonce        = os.urandom(32).hex()

    print(f'  [x402] paying ${amount/1e6:.4f} USDC for {endpoint}')

    signed = account.sign_message(encode_structured_data({
        'types': {
            'EIP712Domain': [
                {'name': 'name',              'type': 'string'},
                {'name': 'version',           'type': 'string'},
                {'name': 'chainId',           'type': 'uint256'},
                {'name': 'verifyingContract', 'type': 'address'},
            ],
            'TransferWithAuthorization': [
                {'name': 'from',        'type': 'address'},
                {'name': 'to',          'type': 'address'},
                {'name': 'value',       'type': 'uint256'},
                {'name': 'validAfter',  'type': 'uint256'},
                {'name': 'validBefore', 'type': 'uint256'},
                {'name': 'nonce',       'type': 'bytes32'},
            ],
        },
        'primaryType': 'TransferWithAuthorization',
        'domain': {'name': 'USD Coin', 'version': '2', 'chainId': 8453, 'verifyingContract': asset},
        'message': {
            'from': account.address, 'to': pay_to,
            'value': amount, 'validAfter': valid_after, 'validBefore': valid_before,
            'nonce': bytes.fromhex(nonce),
        },
    }))

    x_payment = json.dumps({
        'x402Version': 2, 'scheme': 'exact', 'network': 'eip155:8453',
        'payload': {
            'signature': signed.signature.hex(),
            'from': account.address, 'to': pay_to, 'value': str(amount),
            'validAfter': str(valid_after), 'validBefore': str(valid_before),
            'nonce': '0x' + nonce,
        },
    })

    r2 = httpx.get(url, headers={'X-Payment': x_payment}, timeout=15)
    return r2.json() if r2.status_code == 200 else {'error': f'{r2.status_code}', 'body': r2.text[:200]}


print('buy_intelligence() ready')

## Buy signals directly

In [ ]:
# Macro stress — $0.15 USDC
macro = buy_intelligence('/v1/intelligence/macro-stress')
print(json.dumps(macro, indent=2))

In [ ]:
# FX settlement corridor — $0.25 USDC
fx = buy_intelligence('/v1/intelligence/fx-settlement')
print(json.dumps(fx, indent=2))

In [ ]:
# Cascade simulation — $0.75 USDC (most comprehensive)
cascade = buy_intelligence('/v1/intelligence/cascade')
print(json.dumps(cascade, indent=2))

## Use with Claude — intelligence-driven reasoning

Claude can call `buy_intelligence` as a tool, choosing which signals are worth purchasing based on context. This pattern works for any analytical task — not just settlement.

In [ ]:
intel_tool = {
    'name': 'buy_intelligence',
    'description': (
        'Buy a DPX Intelligence signal via x402 micropayment ($0.15–$0.75 USDC on Base mainnet). '
        'Returns structured data on macro conditions, FX corridors, climate, sovereign debt, '
        'cascade risk, supply chain, shipping, or any of the 22 available endpoints. '
        'Each call is a discrete payment — only buy signals relevant to the question.'
    ),
    'input_schema': {
        'type': 'object',
        'properties': {
            'endpoint': {
                'type': 'string',
                'description': 'Intelligence endpoint path, e.g. /v1/intelligence/macro-stress',
                'enum': [
                    '/v1/intelligence/macro-stress',
                    '/v1/intelligence/climate',
                    '/v1/intelligence/climate-pulse',
                    '/v1/intelligence/earth-systems',
                    '/v1/intelligence/supply-chain',
                    '/v1/intelligence/energy-transition',
                    '/v1/intelligence/cascade',
                    '/v1/intelligence/instability',
                    '/v1/intelligence/commodity',
                    '/v1/intelligence/sovereign-debt',
                    '/v1/intelligence/water-risk',
                    '/v1/intelligence/mycelium',
                    '/v1/intelligence/currency-stress',
                    '/v1/intelligence/biodiversity',
                    '/v1/intelligence/shipping-stress',
                    '/v1/intelligence/fx-settlement',
                    '/v1/intelligence/tectonic',
                    '/v1/intelligence/resonance',
                    '/v1/intelligence/gender-risk',
                ]
            },
            'reason': {
                'type': 'string',
                'description': 'Why this signal is relevant to the current question'
            }
        },
        'required': ['endpoint', 'reason']
    }
}


def ask_with_intelligence(question: str):
    """Ask Claude a question. It will buy relevant intelligence signals to answer it."""
    print(f'Question: {question}\n' + '='*60)
    messages = [{'role': 'user', 'content': question}]
    system = (
        'You have access to DPX Intelligence — a paid signal API covering macro conditions, '
        'climate systems, FX corridors, cascade risk, supply chain, sovereign debt, and more. '
        'Each signal costs $0.15–$0.75 USDC from your wallet. Only buy signals that are '
        'genuinely relevant to the question. Explain what each signal tells you and '
        'how it shapes your answer.'
    )
    while True:
        resp = client.messages.create(
            model='claude-sonnet-5', max_tokens=2048,
            system=system, tools=[intel_tool], messages=messages,
        )
        for b in resp.content:
            if b.type == 'text' and b.text.strip():
                print(f'\nClaude: {b.text}')
        if resp.stop_reason == 'end_turn':
            return
        calls = [b for b in resp.content if b.type == 'tool_use']
        if not calls:
            break
        results = []
        for c in calls:
            print(f'\n  → buy_intelligence({c.input["endpoint"]})')
            print(f'    reason: {c.input["reason"]}')
            out = json.dumps(buy_intelligence(c.input['endpoint']))
            print(f'  ← {out[:100]}...')
            results.append({'type': 'tool_result', 'tool_use_id': c.id, 'content': out})
        messages += [{'role': 'assistant', 'content': resp.content}, {'role': 'user', 'content': results}]


print('ready')

In [ ]:
# Example 1: pure intelligence query — no settlement involved
ask_with_intelligence(
    'What are the top macro and climate risks to monitor over the next 30 days? '
    'Buy the signals you need to give a well-grounded answer.'
)

In [ ]:
# Example 2: supply chain research
ask_with_intelligence(
    'We source goods from Southeast Asia. What supply chain and shipping risks '
    'should we be aware of right now? What signals are most relevant?'
)

## Optional: using intelligence alongside DPX Settlement

DPX Settlement is a separate product — a payment execution rail on Base mainnet. The example below shows how the two products can be used together: buy intelligence signals, then optionally use that context when executing a settlement through DPX. Neither product requires the other.

- **DPX Intelligence**: `intelligence.untitledfinancial.com` — signal API, x402-gated
- **DPX Settlement**: `agent.untitledfinancial.com` — payment execution, oracle-gated

In [ ]:
import httpx

ORACLE  = 'https://stability.untitledfinancial.com'
AGENT   = 'https://agent.untitledfinancial.com'
SANDBOX = True  # set False only with a real funded wallet and real counterparty


def check_oracle() -> dict:
    """DPX Settlement — oracle gate (free)."""
    r = httpx.get(f'{ORACLE}/reliability', timeout=10).json()
    return {
        'status':    r.get('stability', {}).get('latestStatus', 'UNKNOWN'),
        'score':     r.get('stability', {}).get('currentScore', 0),
        'reasoning': r.get('intelligence', {}).get('reasoning', ''),
    }


def get_quote(amount_usd: float, has_fx=False, esg_score=75) -> dict:
    """DPX Settlement — binding fee quote (free)."""
    r = httpx.get(f'{ORACLE}/quote',
        params={'amountUsd': amount_usd, 'hasFx': str(has_fx).lower(), 'esgScore': esg_score},
        timeout=10).json()
    return {
        'quote_id': r.get('quoteId'),
        'fee_usd':  r.get('fees', {}).get('total', {}).get('usd'),
        'net_usd':  r.get('settlement', {}).get('netUsd'),
    }


def screen(amount: float, recipient: str) -> dict:
    """DPX Settlement — AML/sanctions screen (free)."""
    r = httpx.get(f'{AGENT}/flow-check',
        params={'amount': amount, 'from': 'USD', 'to': 'USD', 'recipientAddress': recipient},
        timeout=15).json()
    return {'decision': r.get('decision', 'BLOCKED'), 'reason': r.get('holdReason') or r.get('blockReason') or 'OK'}


def settle(amount: float, recipient: str, quote_id: str, purpose: str) -> dict:
    """DPX Settlement — execute."""
    r = httpx.post(f'{AGENT}/settle', json={
        'amount': amount, 'sourceCurrency': 'USD', 'destinationCurrency': 'USD',
        'recipientAddress': recipient, 'purpose': purpose,
        'quoteId': quote_id, 'sandbox': SANDBOX,
    }, timeout=30).json()
    return {'status': r.get('status'), 'settlementId': r.get('settlementId'), 'txHash': r.get('txHash')}


print('Settlement tools ready')

In [ ]:
# Buy relevant intelligence signals first
macro    = buy_intelligence('/v1/intelligence/macro-stress')
fx       = buy_intelligence('/v1/intelligence/fx-settlement')

print('\n── Macro stress ──')
print(f"  Regime: {macro.get('regime')}  Score: {macro.get('score')}")
print(f"  {macro.get('recommendation')}")

print('\n── FX corridor ──')
print(f"  Stability: {fx.get('stability')}  Execution risk: {fx.get('executionRisk')}")
print(f"  {fx.get('recommendation')}")

# Optionally use that context to decide whether to proceed with settlement
if macro.get('regime') != 'CRISIS' and fx.get('executionRisk') in ('LOW', 'MINIMAL'):
    print('\n── Proceeding to settlement ──')
    oracle = check_oracle()
    print(f"  Oracle: {oracle['status']}  Score: {oracle['score']}")

    if oracle['status'] != 'UNSTABLE':
        quote = get_quote(amount_usd=500_000, has_fx=False)
        print(f"  Quote: fee ${quote['fee_usd']}, net ${quote['net_usd']}")

        compliance = screen(500_000, '0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045')
        print(f"  Compliance: {compliance['decision']}")

        if compliance['decision'] == 'PROCEED':
            result = settle(500_000, '0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045',
                            quote['quote_id'], 'Q2 procurement invoice #INV-2026-0088')
            print(f"  Settlement: {result}")
        else:
            print(f"  Held: {compliance['reason']}")
    else:
        print('  Oracle UNSTABLE — settlement held')
else:
    print('\nIntelligence signals indicate adverse conditions — settlement deferred')

## What each product is

| | DPX Intelligence | DPX Settlement |
|---|---|---|  
| **What it does** | Returns structured signals on macro, climate, FX, cascade, sovereign debt, and more | Executes cross-border stablecoin payments, oracle-gated and compliance-screened |
| **Endpoint** | `intelligence.untitledfinancial.com` | `agent.untitledfinancial.com` |
| **Auth** | x402 micropayment per signal | None for quotes/screens; execution is on-chain sender-funded |
| **Price** | $0.15–$0.75 USDC per signal | Basis points on notional |
| **Use cases** | Research, risk management, agent decision-making, data pipelines | Autonomous cross-border payment execution |
| **Requires the other?** | No | No |

## Next steps

- [DPX Intelligence docs](https://docs.untitledfinancial.com/guides/x402-intelligence) — all 22 endpoints, pricing, webhook subscriptions
- [Intelligence API reference](https://docs.untitledfinancial.com/api/intelligence-api) — complete response schemas  
- [DPX Settlement quickstart](https://docs.untitledfinancial.com/agent-quickstart) — oracle gate, fee quote, compliance screen, execute
- [MCP server](https://docs.untitledfinancial.com/integrations/mcp) — 78 tools for Claude Desktop and Cursor
- **Go live**: set `AGENT_PRIVATE_KEY` in `.env` with a funded Base wallet to pay real x402 fees